# Notebook 02 - Build SQL Data Mart (SQLite)

In this notebook we:

1. Load the processed wide dataset (Silver layer)
2. Filter to country-only entities
3. Design a simple star schema:
   - dim_country
   - dim_year
   - fact_country_economics
4. Create a SQLite database
5. Insert data into dimension and fact tables
6. Validate the data mart with SQL queries

This database will serve as the backend for Power BI and SQL analytics.

In [9]:
# Import pandas for data manipulation and DataFrame handling
import pandas as pd

# sqlite3 is built into Python and allows us to create and query SQLite databases
import sqlite3

# pathlib helps us work with file paths in a clean, cross-platform way
from pathlib import Path

## Step 1 - Load the Processed (Silver) Dataset

We load the wide-format dataset created in Notebook 01.

This dataset has:
- one row per (country, year)
- one column per metric (GDP, inflation, etc.)

This will become the source for our SQL data mart.

In [10]:
# Define the path to the processed wide dataset
DATA_PATH = Path("../data_processed/world_bank_indicators_wide.csv")

# Load the CSV file into a pandas DataFrame
wide = pd.read_csv(DATA_PATH)

# Confirm it loaded correctly
print("Loaded wide dataset shape:", wide.shape)

# Preview first few rows
wide.head()

Loaded wide dataset shape: (6525, 8)


,country_iso3,country,year,gdp_per_capita_usd,gdp_usd,inflation_pct,population,unemployment_pct
0,ABW,Aruba,2000,20681.023027,1.873453e+09,4.044021,90588.0,NaN
1,ABW,Aruba,2001,20740.132583,1.896457e+09,2.883604,91439.0,NaN
2,ABW,Aruba,2002,21307.248251,1.961844e+09,3.315247,92074.0,NaN
3,ABW,Aruba,2003,21949.485996,2.044112e+09,3.656365,93128.0,NaN
4,ABW,Aruba,2004,23700.631990,2.254831e+09,2.529129,95138.0,NaN


## Step 2 - Remove World Bank Aggregates

Some World Bank regional aggregates have 3-letter ISO codes
(e.g., AFE, AFW), so they were not removed in Notebook 01.

For the SQL data mart, we explicitly remove known aggregate ISO3 codes
to ensure we keep only sovereign countries.

This ensures:
- Clean analytics
- No double counting
- Accurate BI dashboards

In [11]:
# Curated list of known World Bank aggregate ISO3 codes.
# These represent regions, income groups, or world totals — NOT sovereign countries.
AGG_ISO3 = {
    "AFE","AFW","ARB","CEB","CSS","EAP","EAR","EAS","ECA","ECS","EMU","EUU","FCS",
    "HIC","IBD","IBT","IDX","IDB","IDA","INX","LAC","LCN","LDC","LIC","LMC","LMN",
    "LTE","MEA","MIC","MNA","NAC","OED","OSS","PRE","PST","SAS","SSA","SSF","SST",
    "TEA","TEC","TLA","TMN","TSA","TSS","UMC","WLD"
}

# Remove rows where ISO3 is in the aggregate list
wide_country = wide[~wide["country_iso3"].isin(AGG_ISO3)].copy()

# Print shape comparison
print("Original shape:", wide.shape)
print("Country-only shape:", wide_country.shape)

# Confirm number of unique countries
print("Unique countries:", wide_country["country_iso3"].nunique())

Original shape: (6525, 8)
Country-only shape: (5500, 8)
Unique countries: 220


## Step 3 - Create SQLite Database

SQLite stores the entire database in a single file.

We create:
    db/worldbank.db

If it does not exist, SQLite will automatically create it.

In [12]:
# Ensure a folder exists to store database files
Path("../db").mkdir(exist_ok=True)

# Define the SQLite database file path
DB_PATH = Path("../db/worldbank.db")

# Connect to the SQLite database (creates file if it doesn't exist)
conn = sqlite3.connect(DB_PATH)

# Create cursor for executing SQL statements
cur = conn.cursor()

# Enable foreign key constraints
cur.execute("PRAGMA foreign_keys = ON;")

print("Connected to SQLite DB at:", DB_PATH.resolve())

Connected to SQLite DB at: /mnt/c/Users/glopp/Python-Projects/world-bank-data-mart-powerbi/db/worldbank.db


## Step 4 - Drop Existing Tables (Development Safety)

During development, we may re-run this notebook.

Dropping tables first ensures:
- No duplicate insert errors
- Clean rebuild of schema

In [ ]:
# Drop existing tables if they exist
cur.executescript("""
DROP TABLE IF EXISTS fact_country_economics;
DROP TABLE IF EXISTS dim_country;
DROP TABLE IF EXISTS dim_year;
""")

# Commit changes
conn.commit()

print("Dropped existing tables!")

Dropped existing tables ✅


## Step 5 - Create Star Schema Tables

We build a simple star schema:

Dimension tables:
- dim_country
- dim_year

Fact table:
- fact_country_economics

Primary key:
- (country_iso3, year)

Foreign keys ensure referential integrity.

In [15]:
# Create star schema tables
cur.executescript("""
CREATE TABLE dim_country (
    country_iso3 TEXT PRIMARY KEY,
    country_name TEXT NOT NULL
);

CREATE TABLE dim_year (
    year INTEGER PRIMARY KEY
);

CREATE TABLE fact_country_economics (
    country_iso3 TEXT NOT NULL,
    year INTEGER NOT NULL,
    gdp_usd REAL,
    gdp_per_capita_usd REAL,
    inflation_pct REAL,
    unemployment_pct REAL,
    population REAL,
    PRIMARY KEY (country_iso3, year),
    FOREIGN KEY (country_iso3) REFERENCES dim_country(country_iso3),
    FOREIGN KEY (year) REFERENCES dim_year(year)
);
""")

conn.commit()

print("Created star schema tables!")

Created star schema tables!


## Step 6 - Build Dimension Tables in pandas

We create:
- dim_country from unique ISO3 + country name
- dim_year from unique years

Then we insert them into SQLite.

In [16]:
# Create country dimension
dim_country_df = (
    wide_country[["country_iso3", "country"]]
    .drop_duplicates()
    .rename(columns={"country": "country_name"})
    .sort_values("country_iso3")
    .reset_index(drop=True)
)

# Create year dimension
dim_year_df = (
    wide_country[["year"]]
    .drop_duplicates()
    .sort_values("year")
    .reset_index(drop=True)
)

print("dim_country rows:", len(dim_country_df))
print("dim_year rows:", len(dim_year_df))

dim_country rows: 220
dim_year rows: 25


In [ ]:
# Insert country dimension into SQLite
dim_country_df.to_sql("dim_country", conn, if_exists="append", index=False)

# Insert year dimension into SQLite
dim_year_df.to_sql("dim_year", conn, if_exists="append", index=False)

print("Inserted dimension tables!")

Inserted dimension tables ✅


## Step 7 - Insert Fact Table

The fact table contains numeric economic metrics
for each (country, year).

In [18]:
# Select relevant columns for fact table
fact_df = wide_country[[
    "country_iso3",
    "year",
    "gdp_usd",
    "gdp_per_capita_usd",
    "inflation_pct",
    "unemployment_pct",
    "population"
]].copy()

# Insert into SQLite
fact_df.to_sql("fact_country_economics", conn, if_exists="append", index=False)

print("Inserted fact table rows:", len(fact_df))

Inserted fact table rows: 5500


## Step 8 - Validate Data Mart with SQL Queries

We run SQL queries to confirm:

- Number of countries
- Number of years
- Number of fact rows
- Example analytical query

In [19]:
# Count countries
print(pd.read_sql("SELECT COUNT(*) AS country_count FROM dim_country;", conn))

# Count years
print(pd.read_sql("SELECT COUNT(*) AS year_count FROM dim_year;", conn))

# Count fact rows
print(pd.read_sql("SELECT COUNT(*) AS fact_rows FROM fact_country_economics;", conn))

# Example: Top 5 countries by GDP in 2022
query = """
SELECT d.country_name, f.gdp_usd
FROM fact_country_economics f
JOIN dim_country d ON f.country_iso3 = d.country_iso3
WHERE f.year = 2022
ORDER BY f.gdp_usd DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

   country_count
0            220
   year_count
0          25
   fact_rows
0       5500


,country_name,gdp_usd
0,Low & middle income,3.696530e+13
1,United States,2.560485e+13
2,China,1.831677e+13
3,Japan,4.262463e+12
4,Germany,4.201022e+12
